# tocandira, round 2: train the evaluation net (a sweep per notebook)

After the data round finished. Each run in `RUNS` trains one net on the `DATASET` shards of its
`radius` (missing radii are encoded here from the raw move lists on Drive, once, and
cached), with early stopping on the held out games; the best epoch's `weights.npz` lands in
`models/<SWEEP>/<name>/`. The last cells print a table of every run against nokamute's
static evaluation and plot the validation curves.

One notebook file per parallel sweep: save a copy in Drive per sweep, give each its own
`SWEEP` name and `RUNS`. L4 runtime (about 1.7 units/hour; a run is a few minutes).
Any `Config` field can go in a run: `radius`, `hidden`, `l2`, `lam`, `lr`, `lr_gamma`,
`weight_decay`, `batch`, `patience`, `max_epochs`.

In [ ]:
SWEEP = "j"
DATASET = "m"           # folder under data/ with the move lists (jsonl.gz); "m": relabeled, see 01_relabel
RUNS = [
    {"name": "m-r6-d45-h256", "radius": 6, "hidden": 256, "lam": 1.0, "target": "d45", "patience": 5},
    {"name": "m-r6-d45-h128", "radius": 6, "hidden": 128, "lam": 1.0, "target": "d45", "patience": 5},
]
DATA_ROOT = "/content/drive/MyDrive/tocandira/data"
OUT_ROOT = f"/content/drive/MyDrive/tocandira/models/{SWEEP}"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -e
if ! command -v cargo >/dev/null 2>&1; then curl -sSf https://sh.rustup.rs | sh -s -- -y -q >/dev/null; fi
source "$HOME/.cargo/env"
cd /content
# Cloned to /content/repo, not /content/tocandira: a folder named like the package would shadow it on sys.path.
if [ -d repo ]; then (cd repo && git pull -q && git submodule update --init -q); else git clone -q --recurse-submodules https://github.com/abstratovcm/tocandira repo; fi
cd /content/repo/engine/nokamute
CARGO_PROFILE_RELEASE_LTO=true CARGO_PROFILE_RELEASE_CODEGEN_UNITS=1 cargo build --release -q
# Not an editable install: its .pth file would only be read when the kernel restarts.
cd /content/repo && pip install -q .
./engine/nokamute/target/release/nokamute --help | head -1

In [ ]:
import glob
import os
import subprocess

from tocandira.nnue import Config, load_shards, sweep

raw = sorted(glob.glob(f"{DATA_ROOT}/{DATASET}/*.jsonl.gz"))
datasets = {}
for radius in sorted({run["radius"] for run in RUNS}):
    # The generation notebook encodes at radius 12 into the dataset folder itself.
    folder = f"{DATA_ROOT}/{DATASET}" if radius == 12 else f"{DATA_ROOT}/{DATASET}r{radius}"
    os.makedirs(folder, exist_ok=True)
    for gz in raw:
        npz = f"{folder}/{os.path.basename(gz).replace('.jsonl.gz', '.npz')}"
        if not os.path.exists(npz):
            subprocess.run(["hive-encode", gz, "--radius", str(radius), "--out", npz,
                            "--workers", str(os.cpu_count() or 2)], check=True)
    datasets[radius] = load_shards(sorted(glob.glob(f"{folder}/*.npz")))
    data, meta = datasets[radius]
    print(f"radius {radius}: {meta['shards']} shards ({meta['duplicates']} duplicate files skipped), {len(data['score'])} positions, {meta['num_features']} features")

In [ ]:
configs = [(run["name"], Config(**{k: v for k, v in run.items() if k != "name"})) for run in RUNS]
results = sweep(datasets, configs, OUT_ROOT)

In [ ]:
import json

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for name, _ in configs:
    rows = [json.loads(line) for line in open(f"{OUT_ROOT}/{name}/log.jsonl")]
    epochs = [r for r in rows if "epoch" in r]
    ax.plot([r["epoch"] for r in epochs], [r["loss"] for r in epochs], marker="o", label=name)
ax.axhline(rows[0]["baseline"]["loss"], color="gray", linestyle="--", label="static eval")
ax.set_xlabel("epoch"); ax.set_ylabel("validation loss"); ax.legend(); ax.set_title(f"sweep {SWEEP}")

Reading the table: `loss` is the fit to the blended target on held out games, `pearson`
the agreement with the run's own target, `d5` the agreement with the depth 5 score on the
same held out games (relabeled data only: the column that compares runs with different
targets), `sign` the share of decided positions whose sign matches the winner (saturated
near 63%). Games sharing an opening are held out together. The static eval
row is what 344 lines of hand tuned Rust achieve. The best run's `weights.npz` is what the
engine side loads; the final arbiter is games against nokamute 1.0.3, not this table.